# AKANE E3 本番：資料を根拠に「分からない」と言えるか

HoVer の主張（試しの48件を除く）について、次を集める。網の計算と比べはこちら（Claude）でやる。
1. Gemma-2 2B：資料を見て答え（正しい／間違い）と自信
2. Gemma-2 9B（4ビット）：主張を「隠れた人や物」つきの部分に分ける → 隠れた人や物の候補を資料から出す
3. NLI（DeBERTa大）：段落ごと・文ごとに、元の主張と部分の主張を判定

**途中で切れても大丈夫**：もう一度「すべてのセルを実行」で続きから動く。
計画は plans/E3_evidence_abstain.md（固定、変更は末尾に記録）。

## 0. 準備
1. 「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU**。
2. 鍵マークの `HF_TOKEN`（前と同じもの）にアクセスを許可。
3. 「ランタイム」→「すべてのセルを実行」。

In [ ]:
# ===== 設定 =====
USE_MOCK = False        # True: 作り物のAIで流れだけ確かめる（結果は研究に使わない）
SAVE_TO_DRIVE = True
PER_GROUP = None        # None: 全部（3,952件）。数字にすると「記事数×正解」ごとにその件数だけ選ぶ
GEMMA_ID = "google/gemma-2-2b-it"
GEMMA9_ID = "google/gemma-2-9b-it"
NLI_ID   = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"   # E2-12の「NLI大」と同じ

In [ ]:
import os, json
OUT = "/content/E3_results"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/AKANE_アーキテクチャ/E3_results"
os.makedirs(OUT, exist_ok=True)
if not USE_MOCK:
    import importlib.util, subprocess
    if importlib.util.find_spec("bitsandbytes") is None:   # 入っていない時だけ入れる（他の部品は上げない）
        subprocess.run('pip -q install bitsandbytes --no-deps', shell=True, check=True)
print("保存先:", OUT)

## 1. 道具（データの読み込み・聞き方・本体）

In [ ]:
# E3 試し2回目（Colabのセル本体）。USE_MOCK=True なら作り物のAIで流れだけ確かめる。
# 結果: {OUT}/E3_pilot.json
#   items[i] = {id, claim, label, hops, sents[資料の文], verdict{p_true}, decomp{A:[部分], B:{hidden:{#1:説明}, facts:[...] , cands:{#1:[候補]}}},
#               nli{claim:[[含,中,矛] 文ごと], A:[[...]], B:{"事実の番号|候補の組": [[...]]}}}
import os, re, json, time, random, itertools, collections
import numpy as np

SRC_URL = "https://raw.githubusercontent.com/teacherpeterpan/ProgramFC/main/datasets/HOVER/claims/dev.json"
NAMES = ["entail", "neutral", "contradict"]

def load_hover(path):
    if not os.path.exists(path):
        import urllib.request
        urllib.request.urlretrieve(SRC_URL, path)
    return json.load(open(path))

def pick_pilot(data, per_group=8, seed=0):
    r = random.Random(seed); g = collections.defaultdict(list)
    for e in data: g[(e["num_hops"], e["label"])].append(e)
    out = []
    for k in sorted(g): out += r.sample(g[k], per_group)
    return out

def split_sents(evidence):
    out = []
    for para in evidence.split("\n"):
        para = para.strip()
        if not para: continue
        out += [s.strip() for s in re.split(r'(?<=[.!?])\s+(?=["A-Z0-9])', para) if len(s.strip()) > 3]
    return out

# ---------- Gemmaへの聞き方 ----------
VERDICT_T = ("Evidence:\n{ev}\n\nClaim: {claim}\n\n"
             "Based only on the evidence above, is the claim true or false? Answer with one word: True or False.")

DECOMP_A = """Break the claim into 2 to 4 simple facts. Each fact must be a full sentence that can be checked on its own. Do not add any information that is not in the claim, and do not leave out any part of the claim. Write one fact per line, starting with "- ".

Claim: The author of the novel Kafka on the Shore was born in Kyoto.
- Kafka on the Shore is a novel.
- The author of Kafka on the Shore was born in Kyoto.

Claim: The band that released the album Blue Lines was formed in the same city where the footballer Gary Lineker was born.
- A band released the album Blue Lines.
- The band that released Blue Lines was formed in a city.
- Gary Lineker is a footballer.
- Gary Lineker was born in the city where the band that released Blue Lines was formed.

Claim: {claim}
"""

DECOMP_B = """Rewrite the claim as hidden things and simple facts. A hidden thing is a person, place or thing that the claim describes but does not name. Name hidden things #1, #2, ... and describe each one in a line like "#1 = ...". Use #1, #2 only for things the claim does not name. Then write 2 to 4 simple facts, one per line starting with "- ", using #1, #2 where needed. Do not add any information that is not in the claim, and do not leave out any part of the claim.

Claim: The author of the novel Kafka on the Shore was born in Kyoto.
#1 = the author of the novel Kafka on the Shore
- #1 wrote the novel Kafka on the Shore.
- #1 was born in Kyoto.

Claim: The band that released the album Blue Lines was formed in the same city where the footballer Gary Lineker was born.
#1 = the band that released the album Blue Lines
#2 = the city where #1 was formed
- #1 released the album Blue Lines.
- #1 was formed in #2.
- Gary Lineker was born in #2.

Claim: {claim}
"""

FILL_T = ("Evidence:\n{ev}\n\nUsing only the evidence above, who or what is {desc}? "
          "Answer with its proper name only (the name of the person, place, group or work), not a description. "
          "If the evidence does not say, answer: unknown")
SHOT_WORDS = ["Kafka", "Blue Lines", "Lineker", "Murakami", "Kyoto", "city where #1 was formed"]
UNKNOWN_PAT = re.compile(r"unknown|not (possible|mention|say|provide|tell|relevant)|cannot|can't|does not|doesn't|no mention|impossible|trick question", re.I)

def _leak(line, claim): return any(w in line and w not in claim for w in SHOT_WORDS)
def parse_A(text, claim=""):
    return [l.strip()[2:].strip() for l in text.splitlines() if l.strip().startswith("- ") and len(l.strip()) > 4 and not _leak(l, claim)][:4]

def parse_B(text, claim=""):
    hidden, facts = {}, []
    for l in text.splitlines():
        l = l.strip()
        if _leak(l, claim): continue
        m = re.match(r'^(#\d)\s*=\s*(.+)$', l)
        if m and m.group(1) not in hidden: hidden[m.group(1)] = m.group(2).strip()
        elif l.startswith("- ") and len(l) > 4: facts.append(l[2:].strip())
    facts = [f for f in facts if all(k in hidden for k in re.findall(r'#\d', f))][:4]   # 決めていない #2 などを使う行は捨てる
    used = {k for k in hidden if any(k in f for f in facts)}
    hidden = {k: v for k, v in hidden.items() if k in used}   # 部分の主張で使わない隠れた点は捨てる
    return hidden, facts

def fill(s, assign):
    for k, v in sorted(assign.items(), key=lambda kv: -len(kv[0])): s = s.replace(k, v)
    return s

def clean_name(t):
    t = t.strip().split("\n")[0].strip().strip('."\'*').strip()
    t = re.sub(r'^(the answer is|answer:)\s*', '', t, flags=re.I)
    return "" if UNKNOWN_PAT.search(t) else t[:80]

# ---------- 本体（試し2回目：段階ごと） ----------
def prep(items):
    for it in items:
        it["sents"] = split_sents(it["evidence"])
        it["paras"] = [p.strip() for p in it["evidence"].split("\n") if p.strip()]
    return items

def ev_text(it): return "\n".join(it["sents"])

def do_verdict(items, p_true, log=print):
    pt = p_true([VERDICT_T.format(ev=ev_text(it), claim=it["claim"]) for it in items])
    for it, p in zip(items, pt): it["verdict"] = {"p_true": float(p)}
    log("答えと自信 済み")

def do_decomp(items, gen, tag, log=print, n_samples=3, max_cands=3, max_combos=9):
    """tag: '2b' や '9b'。it['decomp'][tag] に A・B・候補を入れる"""
    ta = gen([DECOMP_A.format(claim=it["claim"]) for it in items])
    tb = gen([DECOMP_B.format(claim=it["claim"]) for it in items])
    for it, a, b in zip(items, ta, tb):
        h, f = parse_B(b, it["claim"])
        it.setdefault("decomp", {})[tag] = {"A_raw": a, "A": parse_A(a, it["claim"]), "B_raw": b,
                                            "B": {"hidden": h, "facts": f, "cands": {}, "unknown": {}}}
    log(f"[{tag}] 分ける 済み")
    for it in items:
        B = it["decomp"][tag]["B"]; assigns = [{}]
        for k in sorted(B["hidden"]):
            new = []
            for a in assigns:
                desc = fill(B["hidden"][k], a)
                pr = [FILL_T.format(ev=ev_text(it), desc=desc)]
                outs = gen(pr, max_new=24) + gen(pr * n_samples, sample=True, max_new=24)
                c, n_unk = [], 0
                for o in outs:
                    o2 = clean_name(o)
                    if not o2: n_unk += 1
                    elif o2 not in c: c.append(o2)
                key = json.dumps(a, ensure_ascii=False)
                B["cands"].setdefault(k, {})[key] = c[:max_cands]
                B["unknown"].setdefault(k, {})[key] = n_unk / len(outs)
                new += [dict(a, **{k: x}) for x in c[:max_cands]]
            assigns = new[:max_combos] or [{}]
        B["assigns"] = assigns
    log(f"[{tag}] 隠れた点の候補 済み")

def do_nli(items, nli, log=print):
    """前提は 文（sents）と 段落（paras）の両方。仮説は 元の主張・各分け方のA・Bに候補を入れたもの"""
    P, H, where = [], [], []
    for i, it in enumerate(items):
        hyps = [("claim", None, it["claim"])]
        for tag, d in it["decomp"].items():
            hyps += [(f"{tag}A", str(fi), f) for fi, f in enumerate(d["A"])]
            for ai, a in enumerate(d["B"]["assigns"]):
                hyps += [(f"{tag}B", f"{fi}|{ai}", fill(f, a)) for fi, f in enumerate(d["B"]["facts"])]
        for unit in ("sents", "paras"):
            for j, s in enumerate(it[unit]):
                for kind, key, h in hyps:
                    P.append(s); H.append(h); where.append((i, unit, kind, key, j))
    log(f"NLIの組 {len(P)}")
    out = nli(P, H)
    for it in items: it["nli"] = {"sents": {}, "paras": {}}
    for (i, unit, kind, key, j), v in zip(where, out):
        d = items[i]["nli"][unit].setdefault(kind, {})
        d.setdefault(key or "_", [None] * len(items[i][unit]))[j] = v
    log("NLI 済み")

# ---------- 作り物のAI（流れの確かめ用。研究には使わない） ----------
def mock_models(seed=0):
    r = np.random.default_rng(seed)
    def gen(prompts, sample=False, max_new=160, **kw):
        out = []
        for p in prompts:
            if p.startswith("Break"):   out.append("- Fact one.\n- Fact two.")
            elif p.startswith("Rewrite"): out.append("#1 = the thing in the claim\n- #1 did something.\n- #1 is somewhere.")
            else: out.append(r.choice(["Alice", "Bob", "Carol"]))
        return out
    def p_true(prompts, **kw): return r.random(len(prompts)).tolist()
    def nli(P, H, **kw): return r.dirichlet([1, 1, 1], len(P)).tolist()
    return gen, p_true, nli

In [ ]:
# E3 本番（Colabのセル本体）。e3_pilot2.py の道具（聞き方・parse・fill など）を使う。
# 途中で切れても、もう一度「すべてのセルを実行」で続きから動く（段階ごと・まとまりごとに保存）。
# 結果: {OUT}/E3_main.json = {"meta":..., "items":[{id, claim, label, num_hops, sents, paras, verdict, decomp9b, nli}]}
import os, json, time

def pick_main(data, exclude_ids, per_group=None, seed=1):
    import random, collections
    rest = [e for e in data if e["id"] not in exclude_ids]
    if per_group is None: return rest
    r = random.Random(seed); g = collections.defaultdict(list)
    for e in rest: g[(e["num_hops"], e["label"])].append(e)
    out = []
    for k in sorted(g): out += r.sample(g[k], per_group)
    return out

def stage_verdict(items, p_true, save, chunk=256, log=print):
    todo = [it for it in items if "verdict" not in it]
    for s in range(0, len(todo), chunk):
        part = todo[s:s + chunk]
        pt = p_true([VERDICT_T.format(ev=ev_text(it), claim=it["claim"]) for it in part])
        for it, p in zip(part, pt): it["verdict"] = {"p_true": float(p)}
        save(); log(f"答えと自信 {s + len(part)}/{len(todo)}")

def stage_decomp(items, gen, save, chunk=256, log=print):
    todo = [it for it in items if "decomp9b" not in it]
    for s in range(0, len(todo), chunk):
        part = todo[s:s + chunk]
        tb = gen([DECOMP_B.format(claim=it["claim"]) for it in part])
        for it, b in zip(part, tb):
            h, f = parse_B(b, it["claim"])
            # 変更2-2：隠れた人や物の説明も確かめる部分の主張にする
            f2 = f + [f"{k} is {v}." for k, v in sorted(h.items())]
            it["decomp9b"] = {"B_raw": b, "hidden": h, "facts": f2, "n_orig_facts": len(f)}
        save(); log(f"分ける {s + len(part)}/{len(todo)}")

def stage_fill(items, gen, save, n_samples=2, max_cands=3, max_assign=4, chunk=256, log=print):
    """隠れた点を順に埋める。まとまりごとに、全部の主張の「次の点」をまとめて聞く"""
    for it in items:
        d = it["decomp9b"]
        if "assigns" not in d:
            d.setdefault("cands", {}); d.setdefault("unknown", {})
            d["_todo"] = sorted(d["hidden"]); d["_assigns"] = [{}]
    def pending(): return [it for it in items if it["decomp9b"].get("_todo")]
    rnd = 0
    while pending():
        rnd += 1; P = pending()
        for s in range(0, len(P), chunk):
            part = P[s:s + chunk]; reqs = []
            for it in part:
                d = it["decomp9b"]; k = d["_todo"][0]
                for a in d["_assigns"]:
                    reqs.append((it, k, a, FILL_T.format(ev=ev_text(it), desc=fill(d["hidden"][k], a))))
            prompts = [q[3] for q in reqs]
            g1 = gen(prompts, max_new=24)
            gs = gen([p for p in prompts for _ in range(n_samples)], sample=True, max_new=24)
            new_assigns = {}
            for qi, (it, k, a, _) in enumerate(reqs):
                outs = [g1[qi]] + gs[qi * n_samples:(qi + 1) * n_samples]
                c, n_unk = [], 0
                for o in outs:
                    o2 = clean_name(o)
                    if not o2: n_unk += 1
                    elif o2 not in c: c.append(o2)
                key = json.dumps(a, ensure_ascii=False); d = it["decomp9b"]
                d["cands"].setdefault(k, {})[key] = c[:max_cands]
                d["unknown"].setdefault(k, {})[key] = n_unk / len(outs)
                new_assigns.setdefault(id(it), []).extend(dict(a, **{k: x}) for x in c[:max_cands])
            for it in part:
                d = it["decomp9b"]; d["_todo"].pop(0)
                d["_assigns"] = new_assigns.get(id(it), [])[:max_assign] or [{}]
            save(); log(f"隠れた点 {rnd}回目 {s + len(part)}/{len(P)}")
    for it in items:
        d = it["decomp9b"]
        if "_assigns" in d: d["assigns"] = d.pop("_assigns"); d.pop("_todo", None)
    save()

def stage_nli(items, nli, save, chunk=128, log=print):
    """変更3：元の主張は文ごと（②数える用）と段落ごと、部分の主張は段落ごと"""
    todo = [it for it in items if "nli" not in it]
    for s in range(0, len(todo), chunk):
        part = todo[s:s + chunk]; P, H, where = [], [], []
        for i, it in enumerate(part):
            d = it["decomp9b"]
            for j, x in enumerate(it["sents"]):
                P.append(x); H.append(it["claim"]); where.append((i, "claim_sents", "_", j))
            for j, x in enumerate(it["paras"]):
                P.append(x); H.append(it["claim"]); where.append((i, "claim_paras", "_", j))
                for ai, a in enumerate(d["assigns"]):
                    for fi, f in enumerate(d["facts"]):
                        P.append(x); H.append(fill(f, a)); where.append((i, "facts_paras", f"{fi}|{ai}", j))
        out = nli(P, H)
        for it in part: it["nli"] = {}
        for (i, kind, key, j), v in zip(where, out):
            it = part[i]; n = len(it["sents"] if kind == "claim_sents" else it["paras"])
            it["nli"].setdefault(kind, {}).setdefault(key, [None] * n)[j] = [round(x, 4) for x in v]
        save(); log(f"NLI {s + len(part)}/{len(todo)}（この回の組 {len(P)}）")

def stage_verdict_drop(items, p_true, save, seed=7, chunk=256, log=print):
    """追加の確かめ：「支える」主張から記事の段落を1つ抜いた資料で、もう一度答えと自信（Gemma 2B）。
    抜いた段落の番号を記録し、NLIの方法は手元で同じ段落の判定を外して計算する。"""
    import random
    r = random.Random(seed)
    for it in items:
        if it["label"] == "supports" and len(it["paras"]) >= 2 and "verdict_drop" not in it:
            it["_drop"] = r.randrange(len(it["paras"]))
    todo = [it for it in items if "_drop" in it and "verdict_drop" not in it]
    for s in range(0, len(todo), chunk):
        part = todo[s:s + chunk]
        ps = []
        for it in part:
            keep = [p for j, p in enumerate(it["paras"]) if j != it["_drop"]]
            ev = "\n".join(x for p in keep for x in split_sents(p))
            ps.append(VERDICT_T.format(ev=ev, claim=it["claim"]))
        pt = p_true(ps)
        for it, p in zip(part, pt): it["verdict_drop"] = {"drop_para": it.pop("_drop"), "p_true": float(p)}
        save(); log(f"1本抜いた答え {s + len(part)}/{len(todo)}")

## 2. AIの道具（読み込みと聞き方）

In [ ]:
import gc
if USE_MOCK:
    gen, p_true, nli = mock_models()
    def load_gemma(mid, four_bit=False): return gen, p_true
    def free(): pass
else:
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, BitsAndBytesConfig
    def batched(fn, xs, bs):
        # bs件ずつ fn に渡す。メモリが足りなければ、その回だけ半分ずつにして続ける
        out, i = [], 0
        while i < len(xs):
            n = bs
            while True:
                try:
                    out += fn(xs[i:i+n]); break
                except torch.cuda.OutOfMemoryError:
                    gc.collect(); torch.cuda.empty_cache()
                    if n == 1: raise
                    n = max(1, n // 2); print("メモリ不足→まとめて聞く数を", n, "に")
            i += n
        return out
    from google.colab import userdata
    HF = userdata.get("HF_TOKEN")
    G = {}
    def free():
        G.clear(); gc.collect(); torch.cuda.empty_cache()
    def load_gemma(mid, four_bit=False):
        free()
        tok = AutoTokenizer.from_pretrained(mid, token=HF, padding_side="left")
        kw = dict(token=HF, device_map="cuda", attn_implementation="eager")
        if four_bit:
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
        else:
            kw["torch_dtype"] = torch.float16
        m = AutoModelForCausalLM.from_pretrained(mid, **kw).eval()
        G["tok"], G["m"] = tok, m
        chat = lambda p: tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
        def gen(prompts, sample=False, bs=8, max_new=160):
            def one(ps):
                enc = tok([chat(p) for p in ps], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
                with torch.no_grad():
                    g = m.generate(**enc, max_new_tokens=max_new, do_sample=sample,
                                   temperature=0.7 if sample else None, top_p=0.95 if sample else None)
                return tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            return batched(one, prompts, bs)
        T_ID = tok.encode("True", add_special_tokens=False)[0]; F_ID = tok.encode("False", add_special_tokens=False)[0]
        def p_true(prompts, bs=8):
            def one(ps):
                enc = tok([chat(p) for p in ps], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
                # 最後の1語の分だけ計算する（全部の語の分を作るとメモリが足りなくなる。答えは同じ）
                with torch.no_grad(): lg = m(**enc, logits_to_keep=1).logits[:, -1, :].float()
                return torch.softmax(lg[:, [T_ID, F_ID]], -1)[:, 0].cpu().tolist()
            return batched(one, prompts, bs)
        return gen, p_true
    def load_nli():
        free()
        ntok = AutoTokenizer.from_pretrained(NLI_ID)
        nm = AutoModelForSequenceClassification.from_pretrained(NLI_ID).cuda().eval()
        G["m"] = nm
        IDX = []
        for name in NAMES:
            hit = [i for i, l in nm.config.id2label.items() if l.lower().startswith(name[:6])]
            assert len(hit) == 1, nm.config.id2label; IDX.append(int(hit[0]))
        def nli(P, H, bs=32):
            def one(ph):
                enc = ntok([a for a, _ in ph], [b for _, b in ph], return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
                with torch.no_grad(): return nm(**enc).logits.float().softmax(-1)[:, IDX].cpu().tolist()
            return batched(one, list(zip(P, H)), bs)
        return nli
print("道具の準備できた")

## 3. 本番を実行（段階ごとに保存。切れたら、もう一度すべてのセルを実行）

In [ ]:
data = load_hover("/content/hover_dev.json")
pilot_ids = {e["id"] for e in pick_pilot(data, per_group=8, seed=0)}
OUTF = f"{OUT}/E3_main.json"
items = pick_main(data, pilot_ids, per_group=PER_GROUP)
if os.path.exists(OUTF):
    old = json.load(open(OUTF))["items"]
    if [o["id"] for o in old] == [it["id"] for it in items]: items = old; print("続きから")
prep(items)
print(len(items), "件")
def save():
    tmp = OUTF + ".tmp"
    json.dump({"meta": {"mock": USE_MOCK, "gemma": GEMMA_ID, "gemma9": GEMMA9_ID, "nli": NLI_ID, "per_group": PER_GROUP},
               "items": items}, open(tmp, "w"), ensure_ascii=False)
    os.replace(tmp, OUTF)
big = (not USE_MOCK) and torch.cuda.get_device_properties(0).total_memory > 20e9
BS = 16 if big else 8
print("GPUが大きい:", big, "まとめて聞く数:", BS)
if any("verdict" not in it for it in items):
    gen, p_true = load_gemma(GEMMA_ID)
    stage_verdict(items, lambda ps: p_true(ps, bs=BS), save)
if any(it["label"] == "supports" and len(it["paras"]) >= 2 and "verdict_drop" not in it for it in items):
    gen, p_true = load_gemma(GEMMA_ID)
    stage_verdict_drop(items, lambda ps: p_true(ps, bs=BS), save)
if any("assigns" not in it.get("decomp9b", {}) for it in items):
    gen, p_true = load_gemma(GEMMA9_ID, four_bit=True)
    g = lambda ps, **kw: gen(ps, bs=BS, **kw)
    stage_decomp(items, g, save)
    stage_fill(items, g, save)
if any("nli" not in it for it in items):
    nli_f = nli if USE_MOCK else load_nli()
    stage_nli(items, lambda P, H: nli_f(P, H, bs=64 if big else 32), save)
print("全部済み:", OUTF)
it = items[0]; print(it["claim"]); print(it["decomp9b"]["hidden"], it["decomp9b"]["facts"], it["decomp9b"]["assigns"])

## 4. ここまで。E3_results の E3_main.json をこちらで読みます。